# Setting I/O Locations

# Environment Setup

In [1]:
!!pip install openpyxl

StatementMeta(, 2bb81215-7256-45bf-a777-bbae1bde8291, 3, Finished, Available, Finished, False)

['Requirement already satisfied: openpyxl in /home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages (3.0.10)',
 'Requirement already satisfied: et_xmlfile in /home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages (from openpyxl) (1.1.0)']

# Importing Date

In [2]:
# List files inside Input folder in Lakehouse

files = mssparkutils.fs.ls("Files")

file_names = [file.name for file in files]



StatementMeta(, 2bb81215-7256-45bf-a777-bbae1bde8291, 4, Finished, Available, Finished, False)

# Importing Files

In [3]:
import pandas as pd

# Define Fabric Lakehouse path
file_path = "/lakehouse/default/Files/RO_Attempts_LST.xlsx"

# Read Excel file
LST = pd.read_excel(
    file_path,
    engine='openpyxl'
)



StatementMeta(, 2bb81215-7256-45bf-a777-bbae1bde8291, 5, Finished, Available, Finished, False)

In [4]:
# Returns all columns for that specific contract
contract_info = LST.loc[LST['Contract'] == 32789670]


StatementMeta(, 2bb81215-7256-45bf-a777-bbae1bde8291, 6, Finished, Available, Finished, False)

In [5]:

import pandas as pd

# Define Fabric Lakehouse path
file_path = "/lakehouse/default/Files/Raw_RO_Attempts.xlsx"

# Read Excel file
Attempts = pd.read_excel(
    file_path,
    engine='openpyxl'
)





StatementMeta(, 2bb81215-7256-45bf-a777-bbae1bde8291, 7, Finished, Available, Finished, False)

In [6]:
Attempts = Attempts.drop_duplicates(subset="Unique_Key", keep="first")

StatementMeta(, 2bb81215-7256-45bf-a777-bbae1bde8291, 8, Finished, Available, Finished, False)

# Making Cartesian Product

In [7]:
LST_copy = LST.copy()
Attempts_copy = Attempts.copy()

StatementMeta(, 2bb81215-7256-45bf-a777-bbae1bde8291, 9, Finished, Available, Finished, False)

In [8]:

# Rename columns (except key) to track origin
LST = LST.rename(columns=lambda x: f"{x}_LST" if x != "Agency_Key" else x)
Attempts = Attempts.rename(columns=lambda x: f"{x}_Attempts" if x != "Agency_Key" else x)



StatementMeta(, 2bb81215-7256-45bf-a777-bbae1bde8291, 10, Finished, Available, Finished, False)

In [9]:
df = (
    LST.assign(_tmp=1)
    .merge(Attempts.assign(_tmp=1), on=["Agency_Key", "_tmp"])
    .drop("_tmp", axis=1)
)

StatementMeta(, 2bb81215-7256-45bf-a777-bbae1bde8291, 11, Finished, Available, Finished, False)

In [10]:
LST.shape

StatementMeta(, 2bb81215-7256-45bf-a777-bbae1bde8291, 12, Finished, Available, Finished, False)

(575402, 31)

In [11]:
Attempts.shape

StatementMeta(, 2bb81215-7256-45bf-a777-bbae1bde8291, 13, Finished, Available, Finished, False)

(537382, 15)

In [12]:
df.shape


StatementMeta(, 2bb81215-7256-45bf-a777-bbae1bde8291, 14, Finished, Available, Finished, False)

(879647, 45)

# Checking Most Frequency Rows


In [13]:
most_common_key = df['Agency_Key'].value_counts().idxmax()


StatementMeta(, 2bb81215-7256-45bf-a777-bbae1bde8291, 15, Finished, Available, Finished, False)

In [14]:
top_rows = df[df['Agency_Key'] == most_common_key]


StatementMeta(, 2bb81215-7256-45bf-a777-bbae1bde8291, 16, Finished, Available, Finished, False)

# Identifying Valid Attempts

In [15]:
# Enforcing date formats

df["Email Date_LST"] = pd.to_datetime(df["Email Date_LST"])
df["Revokation Date_LST"] = pd.to_datetime(df["Revokation Date_LST"])
df["Attempt Date_Attempts"] = pd.to_datetime(df["Attempt Date_Attempts"])

StatementMeta(, 2bb81215-7256-45bf-a777-bbae1bde8291, 17, Finished, Available, Finished, False)

In [16]:
mask = (
    (df["Attempt Date_Attempts"] >= df["Email Date_LST"]) &
    (df["Attempt Date_Attempts"] <= df["Revokation Date_LST"])
)

df["Eligibility"] = "Ineligible"
df.loc[mask, "Eligibility"] = "Eligible"

StatementMeta(, 2bb81215-7256-45bf-a777-bbae1bde8291, 18, Finished, Available, Finished, False)

# Output

In [17]:
# Define Fabric Output path
file_path = "/lakehouse/default/Files/Gold_Output/Eligible_Attempts.csv"

# Save CSV file
df.to_csv(file_path, index=False)

print("Exported successfully to:", file_path)

StatementMeta(, 2bb81215-7256-45bf-a777-bbae1bde8291, 19, Finished, Available, Finished, False)

Exported successfully to: /lakehouse/default/Files/Gold_Output/Eligible_Attempts.csv
